# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.crm_customers")

# Read from bronze

In [0]:
try:
    df = spark.table("workspace.bronze.crm_cust_info_raw")
except Exception as e:
    logger.error(f"Failed to read Bronze table: {e}")
    raise

In [0]:
#df.select([
#    F.count(F.when(F.col(c).isNull(), c)).alias(c)
#    for c in df.columns
#]).show()

#df.describe().show()

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Casting data types

In [0]:
df = (
    df.withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("created_date", F.col("created_date").cast(DateType()))
)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## Dedup, NULL handling and normalization

In [0]:
df = (
    df.dropDuplicates()
    .filter(F.col("customer_id").isNotNull())
    .withColumn(
        "marital_status",
        F.when(F.upper(F.col("marital_status")) == "S", "Single")
        .when(F.upper(F.col("marital_status")) == "M", "Married")
        .otherwise("Unknown")
    )
    .withColumn(
        "gender",
        F.when(F.upper(F.col("gender")) == "M", "Male")
        .when(F.upper(F.col("gender")) == "F", "Female")
        .otherwise("Unknown")
    )
)

# Write into Silver Table

In [0]:
try:
    df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")
except Exception as e:
    logger.error(f"Failed to write Silver table: {e}")
    raise

In [0]:
%sql
SELECT *
FROM workspace.silver.crm_customers